# Interactive EU regional map — 2024

Generate the final static map from the verified analysis table and preserved GISCO
boundaries. This notebook does not download data or change acquisition provenance.
The page has a desktop sidebar, a compact mobile layout, a synchronized color key
and a selected-region panel that also works by touch. It uses one indicator layer
and local Leaflet files; opening the map makes no external requests.


## 1. Locate the project and import the map builder

The Python renderer is in `scripts/build_map.py`. Layout, styling and browser
interactions are kept in `map_assets/`. This notebook provides the workflow for
inspecting the input geography and generating the map.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "data/metadata/acquisition_manifest.json").is_file():
        project_root = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from within the project directory.")

scripts_path = str(project_root / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from build_map import load_map_data, build_map, save_map


## 2. Verify geography and join indicators

The builder verifies the source checksums, NUTS level, CRS, unique analytical keys,
reference codes, indicator bounds and geometry validity before performing a
one-to-one join. Reference regions without unemployment values remain in the map.
Other available GISCO regions are pale, non-interactive geographic context.


In [ ]:
gdf_map, gdf_context = load_map_data(project_root)

geometry_summary = pd.Series({
    "map_regions": len(gdf_map),
    "missing_geometries": gdf_map.geometry.isna().sum(),
    "empty_geometries": gdf_map.geometry.is_empty.sum(),
    "invalid_geometries": (~gdf_map.geometry.is_valid).sum(),
    "missing_unemployment": gdf_map["unemployment_pct"].isna().sum(),
    "missing_gdp": gdf_map["gdp_per_capita_pps"].isna().sum(),
})
display(geometry_summary)
print("Map CRS:", gdf_map.crs)


## 3. Build the interface and synchronized scales

GDP uses blue classes and unemployment uses terracotta classes. Each scale uses
quintile boundaries of available regional values. Missing values are grey. Class
membership uses the original numeric boundaries; only labels are formatted.
Switching indicators updates the same geometry layer, legend and tooltips.
On phones, opening the class key changes the panel layout rather than covering
map content. Selecting a region replaces the mobile legend with its indicator
values; clearing the selection returns to the key.


In [ ]:
labour_map, map_config = build_map(project_root, gdf_map, gdf_context)

for key, metric in map_config["metrics"].items():
    print(key, "available:", metric["available"], "missing:", metric["missing"])
    print("Class boundaries:", metric["edges"])


## 4. Save and inspect the final page

Open `docs/index.html` in a browser. Keep `docs/assets/vendor/` beside it: Leaflet
is local, so no external script, font or tile service is required. No map preview
is embedded in the notebook because relative asset paths belong to the saved page.

Check both metrics, Lubuskie's unavailable unemployment value, the class key,
region selection, Europe/all-regions views and a narrow browser window. Regenerate
with this notebook or `python scripts/build_map.py`; do not patch the generated
HTML directly. See `docs/VALIDATION.md` for verification procedures and their scope.


In [ ]:
map_path = save_map(project_root, labour_map)
print("Map saved:", map_path.relative_to(project_root))
print("HTML size (MB):", round(map_path.stat().st_size / 1_000_000, 2))
